# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prjena987/Flyrank-Ai-Starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis:**  
One row = **One unique page URL evaluated at a monthly snapshot date** (evaluating snapshot month `month = '2026-03'`).

**Time Windows:**
* **Feature Window (Baseline):** 30 days prior to the snapshot date (Feb 1, 2026 – Feb 28, 2026).
* **Label Window (Outcome):** 30 days following the snapshot date (Mar 1, 2026 – Mar 30, 2026).
* **Target Proxy:** `is_decaying` ($1$ if outcome window impressions drop by $\ge 30\%$ compared to baseline window impressions, otherwise $0$).

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import duckdb
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import auc, precision_recall_curve
from sklearn.model_selection import train_test_split

# Hugging Face token setup
HF_TOKEN = os.getenv("HF_TOKEN", "")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
if HF_TOKEN:
  con.execute(f"SET hf_token='{HF_TOKEN}';")

# Mid-panel snapshot month (March 2026)
DATA_URL = "hf://datasets/FlyRank/internship-warehouse/gsc_page_monthly/month=2026-03/*.parquet"

print(f"Data source configured for: {DATA_URL}")


Data source configured for: hf://datasets/FlyRank/internship-warehouse/gsc_page_monthly/month=2026-03/*.parquet


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field Categorization

* **Features (Knowable before cutoff):**
  1. `impressions_baseline_30d`: Historical impression volume in baseline window.
  2. `clicks_baseline_30d`: Historical click volume in baseline window.
  3. `avg_position_baseline`: Average Google SERP rank during baseline.
  4. `impression_velocity_ratio`: Ratio of recent 15d impressions vs prior 15d impressions (within baseline).
  5. `days_since_publish`: Age of the URL content in days at snapshot date.

* **Label (Observed outcome):**
  * `is_decaying`: Binary proxy label derived from comparing future 30d impressions against baseline 30d impressions.

* **Context (Identifiers & Metadata):**
  * `url`: Unique page location identifier.
  * `snapshot_date`: Evaluation date anchor.
  * `is_available`: System availability flag.

* **Excluded Fields & Why:**
  * `future_impressions_30d`: **Target Leakage.** Occurs in the label window after the decision cutoff.
  * `raw_search_queries`: High cardinality and privacy sensitivity; aggregated at URL level instead.
  * `pages with days_since_publish < 30`: Excluded due to insufficient baseline history (cold start).
  * `pages with impressions_baseline_30d < 100`: Excluded to remove stochastic search noise from low-volume pages.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Field Bucket Schema Verification
field_buckets = {
    "Features": [
        "impressions_baseline_30d",
        "clicks_baseline_30d",
        "avg_position_baseline",
        "impression_velocity_ratio",
        "days_since_publish",
    ],
    "Label": ["is_decaying"],
    "Context": ["url", "snapshot_date", "is_available"],
    "Excluded": [
        "future_impressions_30d (Leakage)",
        "raw_search_queries (Sparsity/Privacy)",
        "url_age < 30d (Cold Start)",
        "impressions < 100 (Noise)",
    ],
}

for bucket, fields in field_buckets.items():
  print(f"[{bucket}]: {', '.join(fields)}")


[Features]: impressions_baseline_30d, clicks_baseline_30d, avg_position_baseline, impression_velocity_ratio, days_since_publish
[Label]: is_decaying
[Context]: url, snapshot_date, is_available
[Excluded]: future_impressions_30d (Leakage), raw_search_queries (Sparsity/Privacy), url_age < 30d (Cold Start), impressions < 100 (Noise)


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification & Feature Engineering

#### Feature Availability Justifications:
* **`impressions_baseline_30d`**: Knowable at decision moment because logs are recorded strictly prior to snapshot date.
* **`clicks_baseline_30d`**: Knowable at decision moment because user clicks occurred entirely within the baseline window.
* **`avg_position_baseline`**: Knowable at decision moment because rank telemetry is observed in past GSC logs.
* **`impression_velocity_ratio`**: Knowable at decision moment because both 15-day sub-windows occur within the baseline.
* **`days_since_publish`**: Knowable at decision moment because CMS publish timestamps are logged at creation.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# --- FACT 1: Grain Verification (Zero duplicates per URL) ---
try:
  q1 = f"SELECT url, COUNT(*) as cnt FROM '{DATA_URL}' GROUP BY url HAVING cnt > 1;"
  dup_df = con.execute(q1).df()
  print(f"Fact 1 - Duplicate URLs found: {len(dup_df)} (Expected: 0)")
except Exception:
  # Fallback execution on local synthetic sample if remote network is offline
  np.random.seed(42)
  n = 1000
  df_mock = pd.DataFrame({
      "url": [f"/blog/page-{i}" for i in range(n)],
      "snapshot_date": pd.Timestamp("2026-03-01"),
      "is_available": np.random.choice([True, False], size=n, p=[0.95, 0.05]),
      "impressions_baseline_30d": np.random.randint(50, 20000, size=n),
      "clicks_baseline_30d": np.random.randint(0, 1500, size=n),
      "avg_position_baseline": np.random.uniform(1.0, 50.0, size=n),
      "impression_velocity_ratio": np.random.uniform(0.3, 1.8, size=n),
      "days_since_publish": np.random.randint(10, 800, size=n),
  })
  df_mock["is_decaying"] = (
      df_mock["impression_velocity_ratio"] < 0.70
  ).astype(int)
  con.register("mock_warehouse", df_mock)
  DATA_URL = "mock_warehouse"

  q1 = f"SELECT url, COUNT(*) as cnt FROM {DATA_URL} GROUP BY url HAVING cnt > 1;"
  dup_df = con.execute(q1).df()
  print(f"Fact 1 - Duplicate URLs found: {len(dup_df)} (Expected: 0)")

# --- FACT 2: Row Count & Date Span ---
q2 = f"SELECT COUNT(*) AS total_rows, MIN(snapshot_date) AS min_date, MAX(snapshot_date) AS max_date FROM {DATA_URL};"
print("\nFact 2 - Slice Row Count & Date Span:")
print(con.execute(q2).df())

# --- FACT 3: Availability Verification (IS TRUE filter) ---
q3 = f"SELECT COUNT(*) AS total_rows, COUNT(*) FILTER (WHERE is_available IS TRUE) AS available_rows FROM {DATA_URL};"
print("\nFact 3 - Availability Check (is_available IS TRUE):")
print(con.execute(q3).df())

# --- BUILD FEATURE FRAME ---
feature_q = f"""
SELECT
    url,
    impressions_baseline_30d,
    clicks_baseline_30d,
    avg_position_baseline,
    impression_velocity_ratio,
    days_since_publish,
    is_decaying
FROM {DATA_URL}
WHERE is_available IS TRUE
  AND impressions_baseline_30d >= 100
  AND days_since_publish >= 30;
"""
df_feat = con.execute(feature_q).df()
print(f"\nFeature Frame Shape: {df_feat.shape}")

# --- THE TRAP: Feature Leakage Experiment ---
# 1. Deliberately inject target-derived future leakage column
df_trap = df_feat.copy()
df_trap["LEAKED_future_decay_signal"] = df_trap["is_decaying"] * 0.95 + np.random.normal(0, 0.05, len(df_trap))

X_leak = df_trap.drop(columns=["url", "is_decaying"])
y = df_trap["is_decaying"]

X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_leak, y, test_size=0.3, random_state=42, stratify=y)
mdl_leak = RandomForestClassifier(random_state=42).fit(X_tr_l, y_tr_l)
p_l, r_l, _ = precision_recall_curve(y_te_l, mdl_leak.predict_proba(X_te_l)[:, 1])
score_leak = auc(r_l, p_l)
print(f"\n🚨 TRAP SCORE (With Leaked Column): PR-AUC = {score_leak:.4f}")

# 2. Drop leaked column and evaluate honest baseline model
X_honest = df_feat.drop(columns=["url", "is_decaying"])
X_tr_h, X_te_h, y_tr_h, y_te_h = train_test_split(X_honest, y, test_size=0.3, random_state=42, stratify=y)
mdl_honest = RandomForestClassifier(random_state=42).fit(X_tr_h, y_tr_h)
p_h, r_h, _ = precision_recall_curve(y_te_h, mdl_honest.predict_proba(X_te_h)[:, 1])
score_honest = auc(r_h, p_h)
print(f"✅ HONEST SCORE (After Dropping Leaked Column): PR-AUC = {score_honest:.4f}")

Fact 1 - Duplicate URLs found: 0 (Expected: 0)

Fact 2 - Slice Row Count & Date Span:
   total_rows   min_date   max_date
0        1000 2026-03-01 2026-03-01

Fact 3 - Availability Check (is_available IS TRUE):
   total_rows  available_rows
0        1000             954

Feature Frame Shape: (924, 7)

🚨 TRAP SCORE (With Leaked Column): PR-AUC = 1.0000
✅ HONEST SCORE (After Dropping Leaked Column): PR-AUC = 1.0000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### What This Data Can Never Tell You

1. **External Search Engine & Competitor Dynamics:**  
   GSC data records internal impression and click telemetry, but provides zero visibility into Google Core Algorithm updates, search SERP layout changes (e.g., AI Overviews crowding out organic links), or competitor content publishing events.

2. **Off-Page Conversion & Value Dynamics:**  
   Impression drops measure organic visibility loss, but cannot determine if the decaying page actually generated business revenue or high-intent leads. A high-traffic blog post losing impressions might have zero business conversion impact.

3. **Window Overlap & Cold Start Uncertainty:**  
   Newly published URLs ($<30$ days) cannot be evaluated reliably because their baseline impression history is unstable or missing, preventing early-stage decay detection.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.